In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

pd.set_option('display.max_columns', None)

C:\Users\ARYAN DASH\anaconda3\envs\loan_default\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../data/processed/loan_data_engineered.csv')

In [3]:
drop_cols = ['loan_status', 'default', 'earliest_cr_line']
X = df.drop(columns=drop_cols)
y = df['default']

In [4]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()
for col in categorical_cols:
    X[col] = X[col].astype('category')

C:\Users\ARYAN DASH\AppData\Local\Temp\ipykernel_21740\3967489894.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include='object').columns.tolist()


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
scale_pos_weight_value = y_train.value_counts()[0] / y_train.value_counts()[1]
print(f"scale_pos_weight: {scale_pos_weight_value:.4f}")
print(X_train.shape, X_test.shape)

scale_pos_weight: 4.0088
(1076280, 47) (269070, 47)


In [7]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [8]:
def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'random_state': 42,
        'deterministic': True,
        'force_row_wise': True,
        'scale_pos_weight': scale_pos_weight_value,
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }

    model = lgb.LGBMClassifier(**params)

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(
        model, X_train, y_train, cv=cv, scoring='roc_auc',
        params={'categorical_feature': categorical_cols}
    )

    return scores.mean()

In [9]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10, show_progress_bar=True)

print("Best AUC:", study.best_value)
print("Best params:", study.best_params)

[I 2026-09-11 22:48:13,188] A new study created in memory with name: no-name-1fadc1bc-8d86-49bd-b131-8dc20fb7686d
  0%|          | 0/10 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 143253, number of negative: 574267
[LightGBM] [Info] Total Bins 5642
[LightGBM] [Info] Number of data points in the train set: 717520, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.199650 -> initscore=-1.388482
[LightGBM] [Info] Start training from score -1.388482
[LightGBM] [Info] Number of positive: 143253, number of negative: 574267
[LightGBM] [Info] Total Bins 5648
[LightGBM] [Info] Number of data points in the train set: 717520, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.199650 -> initscore=-1.388482
[LightGBM] [Info] Start training from score -1.388482
[LightGBM] [Info] Number of positive: 143252, number of negative: 574268
[LightGBM] [Info] Total Bins 5644
[LightGBM] [Info] Number of data points in the train set: 717520, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.199649 -> initscore=-1.388491
[LightGBM] [Info] Start training from score -1.38

Best trial: 0. Best value: 0.725415:  10%|█         | 1/10 [01:10<10:33, 70.34s/it]

[I 2026-09-11 22:49:23,528] Trial 0 finished with value: 0.7254151248765872 and parameters: {'num_leaves': 72, 'learning_rate': 0.07046177665942802, 'n_estimators': 234, 'min_child_samples': 79, 'subsample': 0.7666907537055605, 'colsample_bytree': 0.9222361174315155}. Best is trial 0 with value: 0.7254151248765872.
[LightGBM] [Info] Number of positive: 143253, number of negative: 574267
[LightGBM] [Info] Total Bins 5642
[LightGBM] [Info] Number of data points in the train set: 717520, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.199650 -> initscore=-1.388482
[LightGBM] [Info] Start training from score -1.388482
[LightGBM] [Info] Number of positive: 143253, number of negative: 574267
[LightGBM] [Info] Total Bins 5648
[LightGBM] [Info] Number of data points in the train set: 717520, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.199650 -> initscore=-1.388482
[LightGBM] [Info] Start training from score -1.388482
[LightGBM] [I

Best trial: 0. Best value: 0.725415:  20%|██        | 2/10 [02:55<12:05, 90.69s/it]

...

[I 2026-09-11 22:56:49,057] Trial 7 finished with value: 0.7224397968418185 and parameters: {'num_leaves': 22, 'learning_rate': 0.2826868050502669, 'n_estimators': 164, 'min_child_samples': 23, 'subsample': 0.6684396135245024, 'colsample_bytree': 0.7108173094803606}. Best is trial 0 with value: 0.7254151248765872.
[LightGBM] [Info] Number of positive: 143253, number of negative: 574267
[LightGBM] [Info] Total Bins 5642
[LightGBM] [Info] Number of data points in the train set: 717520, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.199650 -> initscore=-1.388482
[LightGBM] [Info] Start training from score -1.388482
[LightGBM] [Info] Number of positive: 143253, number of negative: 574267
[LightGBM] [Info] Total Bins 5648
[LightGBM] [Info] Number of data points in the train set: 717520, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.199650 -> initscore=-1.388482
[LightGBM] [Info] Start training from score -1.388482
[LightGBM] [In

Best trial: 0. Best value: 0.725415:  90%|█████████ | 9/10 [09:41<00:53, 53.65s/it]

[I 2026-09-11 22:57:55,146] Trial 8 finished with value: 0.7244710290045506 and parameters: {'num_leaves': 28, 'learning_rate': 0.06351395340765983, 'n_estimators': 288, 'min_child_samples': 98, 'subsample': 0.8782974962714547, 'colsample_bytree': 0.8652694236759357}. Best is trial 0 with value: 0.7254151248765872.
[LightGBM] [Info] Number of positive: 143253, number of negative: 574267
[LightGBM] [Info] Total Bins 5642
[LightGBM] [Info] Number of data points in the train set: 717520, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.199650 -> initscore=-1.388482
[LightGBM] [Info] Start training from score -1.388482
[LightGBM] [Info] Number of positive: 143253, number of negative: 574267
[LightGBM] [Info] Total Bins 5648
[LightGBM] [Info] Number of data points in the train set: 717520, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.199650 -> initscore=-1.388482
[LightGBM] [Info] Start training from score -1.388482
[LightGBM] [I

Best trial: 0. Best value: 0.725415: 100%|██████████| 10/10 [10:19<00:00, 61.95s/it]

[I 2026-09-11 22:58:32,678] Trial 9 finished with value: 0.7214219930732076 and parameters: {'num_leaves': 53, 'learning_rate': 0.26858236748702263, 'n_estimators': 106, 'min_child_samples': 67, 'subsample': 0.9502782482343273, 'colsample_bytree': 0.7396163875043609}. Best is trial 0 with value: 0.7254151248765872.
Best AUC: 0.7254151248765872
Best params: {'num_leaves': 72, 'learning_rate': 0.07046177665942802, 'n_estimators': 234, 'min_child_samples': 79, 'subsample': 0.7666907537055605, 'colsample_bytree': 0.9222361174315155}


In [10]:
joblib.dump(study, '../models/optuna_study.pkl')
print("Study saved")

print("Best AUC:", study.best_value)
print("Best params:", study.best_params)

Study saved
Best AUC: 0.7254151248765872
Best params: {'num_leaves': 72, 'learning_rate': 0.07046177665942802, 'n_estimators': 234, 'min_child_samples': 79, 'subsample': 0.7666907537055605, 'colsample_bytree': 0.9222361174315155}
